# 02 HEST-1k SF 诊断与下一步模型修改

这个 notebook 记录当前 HIPT MLP size factor predictor 的失败原因诊断。重点不是重新训练，而是把已经跑完的诊断结果整理清楚：模型在哪些器官、哪些队列、哪些切片上有效，在哪些地方失败，以及下一步为什么要改模型。

当前诊断目录：`../results/hest1k_human_visium_sf/diagnostics/sf_main_hipt256_leave_slide_out/`。

## 1. 诊断问题

当前模型已经证明 H&E 图像特征里有 SF 信号，但它还不能作为主结论模型。核心问题有三个：

1. **整体相关性中等**：test `log_sf_pearson` 约为 `0.52`，说明能排出一部分高低趋势，但还不够强。
2. **预测波动偏小**：test `sf_std_ratio` 约为 `0.40`，说明模型预测的 SF 起伏只有真实起伏的 40% 左右。
3. **高 SF 区域低估**：test `sf_top_decile_mean_ratio` 约为 `0.58`，说明真实最高 10% 区域里，模型平均只预测到真实强度的 58% 左右。

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

DIAG = Path("../results/hest1k_human_visium_sf/diagnostics/sf_main_hipt256_leave_slide_out")
assert DIAG.exists(), DIAG

overall = pd.read_csv(DIAG / "metrics_overall_uncalibrated.csv")
overall_cal = pd.read_csv(DIAG / "metrics_overall_val_affine_calibrated.csv")
by_slide = pd.read_csv(DIAG / "metrics_by_slide_uncalibrated.csv")
by_organ = pd.read_csv(DIAG / "metrics_by_organ_uncalibrated.csv")
calibration = pd.read_json(DIAG / "calibration.json", typ="series")

metric_cols = [
    "log_sf_pearson",
    "sf_pearson",
    "log_sf_mae",
    "log_sf_rmse",
    "sf_std_ratio",
    "sf_top_decile_mean_ratio",
    "log_sf_top_decile_mae",
    "n_spots",
]

display(overall[["split"] + metric_cols].round(4))
display(Markdown(f"验证集 affine calibration: `scale={calibration['scale']:.4f}`, `bias={calibration['bias']:.4f}`"))
display(overall_cal[["split"] + metric_cols].round(4))

## 2. 全局校准没有解决问题

验证集 affine calibration 指的是只拟合一个线性校准：`pred_calibrated = a * pred + b`。如果模型只是整体偏大或偏小，这一步应该能改善 test 结果。

但这次校准后的 test 表现变差：

| 设置 | test log SF 相关性 | test log SF MAE | SF 波动恢复比例 | 最高 10% SF 强度恢复比例 |
|---|---:|---:|---:|---:|
| 未校准 | 0.5231 | 0.6606 | 0.4039 | 0.5772 |
| validation affine calibration | 0.5207 | 0.6671 | 0.2689 | 0.4893 |

结论：问题不是一个简单的全局尺度偏差。模型在不同器官、不同队列上学到的规律不一致，因此只靠 `a, b` 校准会把某些切片进一步压平。

In [ ]:
test_organs = by_organ[by_organ["split"] == "test"].copy()
organ_view = test_organs.sort_values("log_sf_pearson")[[
    "organ",
    "log_sf_pearson",
    "log_sf_mae",
    "sf_std_ratio",
    "sf_top_decile_mean_ratio",
    "n_spots",
]]
display(organ_view.round(4))

## 3. 失败主要集中在特定器官

test set 的器官级结果显示：

- **表现较好**：Breast、Bowel、Skin、Eye、Heart。
- **明显失败**：Liver、Pancreas、Cervix。
- **中间状态**：Kidney、Lung、Prostate、Brain、Ovary、Uterus。

这说明 H&E 到 SF 的映射不是一个完全统一的单一规律。不同器官的组织结构、测序平台版本、样本制备方式和图像质量会改变目标分布。后续模型修改必须把切片内空间上下文和队列级差异纳入诊断，而不是只看一个总体 test 相关性。

In [ ]:
test_slides = by_slide[by_slide["split"] == "test"].copy()

worst_slides = test_slides.sort_values("log_sf_pearson").head(12)[[
    "sample_id",
    "organ",
    "cohort",
    "log_sf_pearson",
    "log_sf_mae",
    "sf_std_ratio",
    "sf_top_decile_mean_ratio",
    "n_spots",
]]

best_slides = test_slides.sort_values("log_sf_pearson", ascending=False).head(8)[[
    "sample_id",
    "organ",
    "cohort",
    "log_sf_pearson",
    "log_sf_mae",
    "sf_std_ratio",
    "sf_top_decile_mean_ratio",
    "n_spots",
]]

display(Markdown("### 最差 test slides"))
display(worst_slides.round(4))
display(Markdown("### 最好 test slides"))
display(best_slides.round(4))

## 4. 空间叠图诊断

每张空间诊断图包含四列：

1. `true log SF`：真实的 log size factor 空间分布。
2. `pred log SF`：模型预测的 log size factor 空间分布。
3. `calibrated pred log SF`：验证集 affine calibration 后的预测。
4. `residual pred - true`：预测减真实，红蓝越明显代表误差越大。

下面放两个代表：`TENX91` 是成功案例，`NCBI833` 是失败案例。

In [ ]:
for slide_id in ["TENX91", "NCBI833"]:
    image_path = DIAG / "plots" / "spatial" / f"{slide_id}_sf_spatial_diagnostics.png"
    display(Markdown(f"### {slide_id}"))
    display(Image(filename=str(image_path)))

## 5. 当前判断

当前模型的问题不是“完全没信号”，而是“有信号但不稳”。成功切片说明 HIPT patch 特征能捕捉到一部分组织区域的测序强度变化；失败切片说明单个 spot patch 的局部图像特征不够，尤其不能恢复某些器官里强烈的 SF 尾部区域。

所以下一步模型修改不应该只是调学习率或加深 MLP，而应该先补两类信息：

1. **空间上下文特征**：每个 spot 不只看自己的 HIPT patch，还看附近 spot 的 HIPT 均值、差异和空间邻域结构。
2. **切片内分布约束**：训练时显式惩罚预测被压平，尤其惩罚真实高 SF 区域的系统性低估。

这两个修改对应当前最大错误：`sf_std_ratio` 太低、`sf_top_decile_mean_ratio` 太低。

## 6. 下一步实验

下一步不覆盖现有 `features.npy`，而是创建新的上下文特征文件和新的 manifest/config，保证原始 HIPT 结果可以复现。

计划如下：

1. 为每张 slide 生成 `features_context.npy`：拼接原始 HIPT feature、空间邻居均值、当前 spot 与邻居均值的差、邻域方差，以及归一化坐标。
2. 生成 `human_visium_sf_manifest_context.csv`：只把 `features_path` 指向新的上下文特征，其他 counts、coords、size factor 保持不变。
3. 训练 context SF predictor：使用同一个 leave-slide-out split，不能随机 spot split。
4. 用同一套 diagnostics 脚本比较：重点看 test `log_sf_pearson`、`sf_std_ratio`、`sf_top_decile_mean_ratio` 是否改善。
5. 如果 context 模型改善，再继续做 leave-cohort-out 和 leave-organ-out；如果没有改善，继续检查失败器官的 target 分布和数据质量。

## 7. 已完成的模型修改与结果

本轮已经完成三类模型修改：

1. `context`：加入空间邻域 HIPT 均值和坐标/密度特征。
2. `context_slide_balanced`：在 context 特征基础上使用 slide-balanced sampling。
3. `context_distribution` / `context_distribution_light`：在 context 特征基础上加入分布宽度约束，避免预测过度压平。

test set 结果如下：

| 模型 | log SF 相关性 | log SF MAE | SF 波动恢复比例 | 最高 10% SF 强度恢复比例 | 判断 |
|---|---:|---:|---:|---:|---|
| 原始 HIPT MLP | 0.5231 | 0.6606 | 0.4039 | 0.5772 | 有信号，但不够 |
| context | 0.5864 | 0.6080 | 0.4072 | 0.6108 | 平均误差最好 |
| context + slide-balanced | 0.5434 | 0.6572 | 0.4044 | 0.5753 | 不采用 |
| context + distribution | 0.5809 | 0.6371 | 0.4486 | 0.6374 | 尾部恢复最好 |
| context + light distribution | 0.5878 | 0.6212 | 0.4154 | 0.6172 | 当前综合候选 |

结论：空间 context 是有效修改；slide-balanced sampling 没有帮助；分布约束能改善尾部和波动，但权重要轻。当前可以把 `context + light distribution` 作为下一轮候选模型，同时保留普通 `context` 作为平均误差更低的对照。

In [ ]:
experiment_dirs = {
    "main_hipt_mlp": DIAG,
    "context": Path("../results/hest1k_human_visium_sf/diagnostics/sf_context_hipt256_leave_slide_out"),
    "context_slide_balanced": Path("../results/hest1k_human_visium_sf/diagnostics/sf_context_slide_balanced_hipt256_leave_slide_out"),
    "context_distribution": Path("../results/hest1k_human_visium_sf/diagnostics/sf_context_distribution_hipt256_leave_slide_out"),
    "context_distribution_light": Path("../results/hest1k_human_visium_sf/diagnostics/sf_context_distribution_light_hipt256_leave_slide_out"),
}

rows = []
for name, path in experiment_dirs.items():
    df = pd.read_csv(path / "metrics_overall_uncalibrated.csv")
    test = df[df["split"] == "test"].iloc[0].copy()
    test["model"] = name
    rows.append(test)

comparison = pd.DataFrame(rows)[[
    "model",
    "log_sf_pearson",
    "sf_pearson",
    "log_sf_mae",
    "log_sf_rmse",
    "sf_std_ratio",
    "sf_top_decile_mean_ratio",
    "log_sf_top_decile_mae",
    "n_spots",
]]
display(comparison.round(4))

## 8. 仍然没有解决的问题

即使换成 context + light distribution，Liver、Pancreas、Cervix 仍然失败。它们不是靠一个更深 MLP 就能解决的问题。下一步必须单独检查这些失败器官：

1. target 分布是否异常，比如 Liver 的真实 log SF 范围过大。
2. patch 覆盖和 valid spot 是否存在系统性偏差。
3. 是否存在队列或技术差异，比如 FFPE、fresh frozen、targeted panel、CytAssist 等。
4. 是否需要器官内模型、器官条件模型，或在最终 H&E-to-ST 任务里使用 organ-aware calibration。

当前不能宣称已经有顶刊级主模型。可以宣称的是：我们找到了有效改动，context 特征稳定提高 test 相关性；但跨器官稳定性还没过关。